<a href="https://colab.research.google.com/github/Dhavalkumar510/Fortray-Project/blob/main/workshop_for_fortray.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Work Shop for Fortray

### Importing necessary libraries

In [ ]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

## Step 1: Robust Data Ingestion & Transformation

In [ ]:
df = pd.read_csv("hm_land_registry_2000.csv")

In [ ]:
# Dataset overview
print("Dataset Shape:")
print(df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

Dataset Shape:
(2000, 8)

Column Names:
['Transaction_Date', 'Region', 'Property_Type', 'EPC_Rating', 'Days_on_Market', 'Asking_Price', 'Sold_Price', 'Market_Status']

Data Types:
Transaction_Date    object
Region              object
Property_Type       object
EPC_Rating          object
Days_on_Market       int64
Asking_Price         int64
Sold_Price           int64
Market_Status       object
dtype: object


In [ ]:
print("\nMissing Values Per Column:")
print(df.isnull().sum())

print("\nPercentage Missing:")
print((df.isnull().sum() / len(df)) * 100)


Missing Values Per Column:
Transaction_Date    0
Region              0
Property_Type       0
EPC_Rating          0
Days_on_Market      0
Asking_Price        0
Sold_Price          0
Market_Status       0
dtype: int64

Percentage Missing:
Transaction_Date    0.0
Region              0.0
Property_Type       0.0
EPC_Rating          0.0
Days_on_Market      0.0
Asking_Price        0.0
Sold_Price          0.0
Market_Status       0.0
dtype: float64


In [ ]:
duplicates = df.duplicated().sum()

print(f"\nDuplicate Rows: {duplicates}")

# Optional removal
df = df.drop_duplicates()


Duplicate Rows: 0


In [ ]:
print("\nNumerical Summary:")
print(df.describe())


Numerical Summary:
       Days_on_Market  Asking_Price    Sold_Price
count     2000.000000  2.000000e+03  2.000000e+03
mean        74.716000  3.049350e+05  2.983974e+05
std         44.288159  1.737615e+05  1.783743e+05
min        -20.000000  8.495800e+04  1.244100e+04
25%         37.750000  1.870295e+05  1.801570e+05
50%         74.000000  2.566665e+05  2.537275e+05
75%        114.000000  3.681950e+05  3.630018e+05
max        149.000000  1.037381e+06  1.075929e+06


In [ ]:
print("\nNegative Asking Prices:")
print((df['Asking_Price'] <= 0).sum())

print("\nNegative Sold Prices:")
print((df['Sold_Price'] <= 0).sum())


Negative Asking Prices:
0

Negative Sold Prices:
0


In [ ]:
print("\nNegative Days on Market:")
print((df['Days_on_Market'] < 0).sum())


Negative Days on Market:
50


In [ ]:
print("\nRegions:")
print(df['Region'].unique())

print("\nMarket Status Values:")
print(df['Market_Status'].unique())


Regions:
['London' 'North West' 'South East' 'Scotland' 'Yorkshire' 'West Midlands'
 'Wales']

Market Status Values:
['Normal' 'Stagnant' 'High Demand' 'System_Error_Null']


In [ ]:
print("\nLargest Sold Prices:")
print(df.nlargest(10, 'Sold_Price')[['Region', 'Sold_Price']])

print("\nSmallest Sold Prices:")
print(df.nsmallest(10, 'Sold_Price')[['Region', 'Sold_Price']])


Largest Sold Prices:
      Region  Sold_Price
1683  London     1075929
1900  London     1057897
1819  London     1033543
1492  London     1027694
410   London     1026684
1909  London     1020940
430   London     1012591
1339  London      994461
1030  London      994431
1766  London      993703

Smallest Sold Prices:
             Region  Sold_Price
1743      Yorkshire       12441
106        Scotland       13003
1594  West Midlands       13133
733   West Midlands       13318
801           Wales       14412
692           Wales       14666
1        North West       16395
300        Scotland       17619
1807     North West       17638
1420       Scotland       17821


In [ ]:
# Converting Transaction_Date to datetime as per requirement
df['Transaction_Date'] = pd.to_datetime(df['Transaction_Date'],
                                        format='%d/%m/%Y', errors='coerce')

# Converting pricing columns to numeric
df['Asking_Price'] = pd.to_numeric(df['Asking_Price'], errors='coerce')
df['Sold_Price'] = pd.to_numeric(df['Sold_Price'], errors='coerce')
df['Days_on_Market'] = pd.to_numeric(df['Days_on_Market'], errors='coerce')

# Removing invalid rows:
df = df.dropna(subset=['Transaction_Date', 'Asking_Price', 'Sold_Price',
    'Days_on_Market'])

print(df.dtypes)
print("\n")
print("-"*85)
print("\n")
print(df.head())

Transaction_Date    datetime64[ns]
Region                      object
Property_Type               object
EPC_Rating                  object
Days_on_Market               int64
Asking_Price                 int64
Sold_Price                   int64
Market_Status               object
dtype: object


-------------------------------------------------------------------------------------


  Transaction_Date      Region  Property_Type EPC_Rating  Days_on_Market  \
0       2025-01-30      London       Detached          D             107   
1       2025-02-08  North West           Flat          A              97   
2       2025-03-29      London  Semi-Detached          C              19   
3       2025-01-17  South East       Detached          A             111   
4       2025-03-13    Scotland  Semi-Detached          F              76   

   Asking_Price  Sold_Price Market_Status  
0       1021987      981706        Normal  
1        163946       16395        Normal  
2        426746      427734

## Step 2: Heuristic Anomaly Filtering (The "Intelligent Filter")

In [ ]:
# Detecting suspicious records

df["Quarantine_Reason"] = ""

# Sold Price less than 20% of Asking Price
df.loc[
    df["Sold_Price"] < 0.20 * df["Asking_Price"],
    "Quarantine_Reason"
] += "Sold Price below 20% of Asking Price; "

# Negative Days on Market
df.loc[
    df["Days_on_Market"] < 0,
    "Quarantine_Reason"
] += "Negative Days on Market; "

# Create anomaly mask
anomaly_mask = df["Quarantine_Reason"] != ""

# Separate datasets
quarantine_df = df[anomaly_mask].copy()
clean_df = df[~anomaly_mask].copy()

# Save quarantine file
quarantine_df.to_csv(
    "quarantined_transactions.csv",
    index=False
)

print(f"Quarantined {len(quarantine_df)} anomalous records.")
print(f"Retained {len(clean_df)} clean records.")

Quarantined 100 anomalous records.
Retained 1900 clean records.


In [ ]:
clean_df = clean_df.drop(columns=["Quarantine_Reason"])

##  Step 3: Standardization & Status Recalculation

In [ ]:
# Recalculating Market_Status based on Days_on_Market
clean_df.loc[clean_df['Days_on_Market'] > 120, 'Market_Status'] = 'Stagnant'

clean_df.loc[clean_df['Days_on_Market'] < 14, 'Market_Status'] = 'High Demand'

clean_df.loc[
    (clean_df['Days_on_Market'] >= 14) &
    (clean_df['Days_on_Market'] <= 120),
    'Market_Status'
] = 'Normal'

print("Market_Status column successfully audited and corrected.")

Market_Status column successfully audited and corrected.


## Step 4: Algorithmic Feature Engineering

In [ ]:
# Creating Price_Variance_Percent
clean_df['Price_Variance_Percent'] = (
    (clean_df['Sold_Price'] - clean_df['Asking_Price'])
    / clean_df['Asking_Price']
) * 100

# ingalculate regional quartiles
regional_quantiles = (
    clean_df.groupby('Region')['Sold_Price']
    .quantile([0.25, 0.50, 0.75])
    .unstack()
)

# Creating Valuation_Band
clean_df['Valuation_Band'] = 'Ultra-Premium'

for region in regional_quantiles.index:

    q1 = regional_quantiles.loc[region, 0.25]
    q2 = regional_quantiles.loc[region, 0.50]
    q3 = regional_quantiles.loc[region, 0.75]

    region_mask = clean_df['Region'] == region

    clean_df.loc[
        region_mask & (clean_df['Sold_Price'] <= q1),
        'Valuation_Band'
    ] = 'Entry-Level'

    clean_df.loc[
        region_mask &
        (clean_df['Sold_Price'] > q1) &
        (clean_df['Sold_Price'] <= q2),
        'Valuation_Band'
    ] = 'Mid-Market'

    clean_df.loc[
        region_mask &
        (clean_df['Sold_Price'] > q2) &
        (clean_df['Sold_Price'] <= q3),
        'Valuation_Band'
    ] = 'Premium'

print("Feature engineering completed successfully.")

# Displaying sample output as cleaned_df
print("\nProcessed Data Preview:")
print(clean_df.head())

# Save final processed dataset
clean_df.to_csv('processed_transactions.csv', index=False)

print("\nProcessed dataset saved as 'processed_transactions.csv'")

Feature engineering completed successfully.

Processed Data Preview:
  Transaction_Date      Region  Property_Type EPC_Rating  Days_on_Market  \
0       2025-01-30      London       Detached          D             107   
2       2025-03-29      London  Semi-Detached          C              19   
3       2025-01-17  South East       Detached          A             111   
5       2025-03-13      London  Semi-Detached          B              25   
6       2025-01-19   Yorkshire       Terraced          F             107   

   Asking_Price  Sold_Price Market_Status  Price_Variance_Percent  \
0       1021987      981706        Normal               -3.941440   
2        426746      427734        Normal                0.231519   
3        487823      497564        Normal                1.996831   
5        488570      491530        Normal                0.605850   
6        255896      267223        Normal                4.426408   

  Valuation_Band  
0  Ultra-Premium  
2     Mid-Market  
3 

In [ ]:
import sqlite3
from sqlalchemy import create_engine

# Define a local SQLite database file name
local_db_file = 'processed_housing_data.db'

# Create a SQLAlchemy engine for SQLite
# The /// indicates a relative path to the database file
sqlite_db_url = f'sqlite:///{local_db_file}'
engine_sqlite = create_engine(sqlite_db_url)

# Define the table name for the SQLite database
# Using the previously defined table_name, or a new one if preferred
sqlite_table_name = table_name if 'table_name' in globals() else 'housing_transactions'

# Write the clean_df DataFrame to the SQLite database file
try:
    clean_df.to_sql(sqlite_table_name, engine_sqlite, if_exists='replace', index=False)
    print(f"DataFrame successfully written to table '{sqlite_table_name}' in local SQLite database: '{local_db_file}'.")
    print("You can download this file from the Colab file browser (folder icon on the left).")
except Exception as e:
    print(f"Error writing DataFrame to SQLite database: {e}")

DataFrame successfully written to table 'housing_transactions' in local SQLite database: 'processed_housing_data.db'.
You can download this file from the Colab file browser (folder icon on the left).


## Additional Step: Creating Dataframe into SQL database

In [ ]:
!pip install psycopg2-binary sqlalchemy